# ARC-AGI-3 submission — jinbo explorer v1

零模型纯符号基线: 预算内新颖度探索(见附带 dataset 里的 `kaggle_agent/`)。
真提交(`KAGGLE_IS_COMPETITION_RERUN`)走网关打隐藏游戏; 平时 Save & Run 用比赛自带的公开环境文件离线跑, 同一条代码路径。


In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
from urllib.request import urlopen

NOTEBOOK_T0 = time.time()
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}

# 🚨必须在任何 arc_agi import 之前钉死: RESET=关卡重置(比赛规则), client 构建时缓存
os.environ["ONLY_RESET_LEVELS"] = "true"

WORKING = Path("/kaggle/working") if Path("/kaggle").is_dir() else Path("out")
WORKING.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("RECORDINGS_DIR", str(WORKING / "recordings"))
print("TRUE_SUBMISSION =", TRUE_SUBMISSION)


## 1. 从比赛 wheelhouse 离线安装 arc-agi 运行时


In [ ]:
WHEELS = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")
if WHEELS.is_dir():
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "--no-index",
         "--no-warn-conflicts", "--disable-pip-version-check",
         "--find-links", str(WHEELS), "arc-agi"],
        stdout=subprocess.DEVNULL,
    )
    print("arc-agi installed from wheelhouse")
else:
    print("no wheelhouse (local dev run), using current environment")


## 2. 定位源码 bundle 并挂到 sys.path

按 marker 文件找, 不写死挂载路径(Kaggle 的挂载点会因 owner/slug 冲突而变)。


In [ ]:
MARKER = "arc3-jinbo-bundle.json"
bundle = None
if Path("/kaggle/input").is_dir():
    for m in Path("/kaggle/input").rglob(MARKER):
        bundle = m.parent
        break
if bundle is None:
    bundle = Path.cwd()  # 本地开发: 在仓库根目录直接跑
    assert (bundle / "kaggle_agent").is_dir(), f"kaggle_agent 不在 {bundle}"
sys.path.insert(0, str(bundle))
print("bundle:", bundle)


## 3. 真提交: 等网关就绪 / 离线: 找公开环境文件目录


In [ ]:
def wait_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline, last = time.monotonic() + timeout_s, ""
    probe = base_url.rstrip("/") + "/api/games"
    while time.monotonic() < deadline:
        try:
            with urlopen(probe, timeout=10) as r:
                if r.status < 500:
                    return
        except Exception as e:
            last = repr(e)
        time.sleep(5)
    raise RuntimeError(f"gateway not ready: {last}")

env_dir = None
if TRUE_SUBMISSION and os.environ.get("ARC_BASE_URL"):
    wait_gateway(os.environ["ARC_BASE_URL"])
    print("gateway ready:", os.environ["ARC_BASE_URL"])
else:
    os.environ.pop("KAGGLE_IS_COMPETITION_RERUN", None)  # 保证走 OFFLINE 分支
    cands = []
    if Path("/kaggle/input").is_dir():
        cands += [p for p in Path("/kaggle/input").rglob("environment_files") if p.is_dir()]
    cands.append(bundle / "environment_files_sample")
    cands.append(Path("environment_files"))  # 本地仓库
    env_dir = next((str(p) for p in cands if Path(p).is_dir()), None)
    assert env_dir, "找不到离线环境文件目录"
    print("offline env_dir:", env_dir)


## 4. 跑全部游戏

预算: 真提交给 8h 墙钟(均分给各游戏); 离线冒烟给小预算快速过一遍。


In [ ]:
from kaggle_agent.run_submission import main

if TRUE_SUBMISSION:
    budget = dict(seconds_per_game=1000.0, max_actions=3000,
                  total_seconds=8 * 3600 - (time.time() - NOTEBOOK_T0))
else:
    budget = dict(seconds_per_game=float(os.environ.get("A3_SECONDS_PER_GAME", 60)),
                  max_actions=int(os.environ.get("A3_MAX_ACTIONS", 600)),
                  total_seconds=float(os.environ.get("A3_TOTAL_SECONDS", 3000)))

summary = main(env_dir=env_dir or "environment_files", out_dir=str(WORKING), **budget)


## 5. 结果一览


In [ ]:
print(json.dumps({k: v for k, v in summary.items() if k != "scorecard"}, indent=2, ensure_ascii=False))
for g in summary["games"]:
    print(f"{g['game_id']:>6}  levels {g['levels_completed']}/{g['win_levels']}"
          f"  steps={g['steps']}  {g['state']}  {g['seconds']}s")
